In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())

print("Click here to download the motion-dataset.zip: ", FileLink("motion-dataset.zip"))


inside dir:  ['requirements.txt', 'networks', 'glove', 'exp_results', 'README.md', 'data_utils', 'dataset_split_builder.py', 'checkpoints', '.git', 'options', 'main.ipynb', 'motion-dataset.zip', 'common', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']
Click here to download the motion-dataset.zip:  /notebooks/motion-synthesis/motion-dataset.zip


In [2]:
import torch
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE
from networks.trainers import MotionVQVAETrainer
from torch.utils.data import Subset
from networks.nn_validator import VQVAEValidator
import json

In [3]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '10000'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.output_dir = options.experiment_dir
options.save_every_e = 100
options.is_continue = False

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain


Device used:  cuda:0


In [ ]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_file = pjoin(options.data_root, 'train_micro.txt')
val_split_file = pjoin(options.data_root, 'val_micro.txt')

if options.dataset_mode == "micro":
    subset_path = os.path.join(options.meta_dir, "micro_subsets.json")
    par_train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    par_val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)

    if os.path.exists(subset_path):
        with open(subset_path, "r") as f:
            subsets = json.load(f)
        micro_train_indices = subsets["micro_train_indices"]
        micro_val_indices = subsets["micro_val_indices"]
    else:
        seed = 42
        rng = np.random.default_rng(seed)

        all_train_indices = np.arange(len(par_train_dataset))
        all_val_indices = np.arange(len(par_val_dataset))

        micro_train_indices = rng.permutation(all_train_indices)[:80].tolist()
        micro_val_indices = rng.permutation(all_val_indices)[:30].tolist()

        with open(subset_path, "w") as f:
            json.dump({
                "seed": seed,
                "micro_train_indices": micro_train_indices,
                "micro_val_indices": micro_val_indices,
            }, f, indent=4)
    train_dataset = Subset(par_train_dataset, micro_train_indices)
    val_dataset = Subset(par_val_dataset, micro_val_indices)
else:
    train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
sample_motion = train_dataset[-1]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['texts'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 8


100%|██████████| 8/8 [00:00<00:00, 6477.69it/s]


Motion shape (B, T, D): (8, 199, 263)
Total number of motions 8, snippets 468
id list 4


100%|██████████| 4/4 [00:00<00:00, 4388.49it/s]

Motion shape (B, T, D): (4, 170, 263)
Total number of motions 4, snippets 495


NameError: name 'train_dataset' is not defined

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                        shuffle=False, pin_memory=True)
vqvae = MotionVQVAE(
    input_dim=Dp_max,
    enc_hidden_dim=1024,
    dec_hidden_dim=1024,
    latent_dim=256,
    num_embeddings=512,
    beta=0.2
)

trainer = MotionVQVAETrainer(options, vqvae = vqvae)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader)


In [ ]:
'''
vqvae_model_dict = torch.load(pjoin(options.model_dir, 'vqvae_beta_0.2.tar'), map_location = options.device)

vqvae.load_state_dict(vqvae_model_dict['vqvae'])

vqvae_validator = VQVAEValidator(
    opt = options,
    vqvae=vqvae,
    val_dataloader = val_loader
)
rows = vqvae_validator.validate()
for row in rows:
    print(row['snippet_id'], row['video_error'])
'''